In [ ]:
#### NOTES 

# Academic calendar generator

# Responsible for:

# generating every day in the academic year
# holidays
# reading weeks
# study days
# exam periods
# week summaries

# Produces:

# Instruction Week
# Reading Week
# Study Day
# Exam Period

# But not:

# MOS3321 Week 4
# Marketing Week 7
# Midterm Week

In [1]:
####### IMPORTS & PATHS
from pathlib import Path
import yaml
import pandas as pd
from datetime import timedelta
from shared.schedule_utils import *
#from data.configs import *
from shared.loaders import * 

BASE_DIR = Path.cwd()

DATA_FILE = BASE_DIR / "data" / "academic_calendar_2026-27.yaml"
OUTPUT_DIR = BASE_DIR / "outputS" / "calendars" / "draft"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_FILE, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config

{'academic_year': '2026-2027',
 'version': 1,
 'status': 'final',
 'last_reviewed': datetime.date(2026, 6, 17),
 'source': 'https://www.westerncalendar.uwo.ca/SessionalDates.cfm',
 'terms': {'fall_2026': {'name': 'Fall 2026',
   'instruction': {'start': datetime.date(2026, 9, 9),
    'end': datetime.date(2026, 12, 9)},
   'study_days': {'start': datetime.date(2026, 12, 10),
    'end': datetime.date(2026, 12, 10)},
   'exams': {'start': datetime.date(2026, 12, 11),
    'end': datetime.date(2026, 12, 22)},
   'breaks': [{'name': 'Fall Reading Week',
     'start': datetime.date(2026, 10, 10),
     'end': datetime.date(2026, 10, 18)}],
   'holidays': [{'name': 'Labour Day', 'date': datetime.date(2026, 9, 7)},
    {'name': 'National Day for Truth and Reconciliation',
     'date': datetime.date(2026, 9, 30)},
    {'name': 'Thanksgiving', 'date': datetime.date(2026, 10, 12)}]},
  'winter_2027': {'name': 'Winter 2027',
   'instruction': {'start': datetime.date(2027, 1, 4),
    'end': datetime.

In [ ]:
####### GENERATE DAY-BY-DAY CALENDAR
rows = []

for term_id, term in config["terms"].items():
    term_start = pd.to_datetime(term["instruction"]["start"])
    exam_end = pd.to_datetime(term["exams"]["end"])

    for date in date_range(term_start, exam_end):
        day_type = "Instruction"
        label = ""

        # Breaks
        for break_item in term.get("breaks", []):
            if pd.to_datetime(break_item["start"]) <= date <= pd.to_datetime(break_item["end"]):
                day_type = "Break"
                label = break_item["name"]

        # Holidays
        for holiday in term.get("holidays", []):
            if date == pd.to_datetime(holiday["date"]):
                day_type = "Holiday"
                label = holiday["name"]

        # Study days
        if "study_days" in term:
            if pd.to_datetime(term["study_days"]["start"]) <= date <= pd.to_datetime(term["study_days"]["end"]):
                day_type = "Study Day"
                label = "Study Day"

        # Exam period
        if pd.to_datetime(term["exams"]["start"]) <= date <= pd.to_datetime(term["exams"]["end"]):
            day_type = "Exam Period"
            label = "Exam Period"

        rows.append({
            "academic_year": config["academic_year"],
            "term_id": term_id,
            "term": term["name"],
            "date": date.date(),
            "day": date.day_name(),
            "week_start": (date - pd.to_timedelta(date.weekday(), unit="D")).date(),
            "day_type": day_type,
            "label": label
        })

calendar_df = pd.DataFrame(rows)
calendar_df.head()

In [ ]:
####### CREATE WEEK SUMMARY 
# Create week summary

####### CREATE WEEK SUMMARY 

planning_df = calendar_df[
    pd.to_datetime(calendar_df["date"]).dt.weekday < 5
].copy()

week_summary = (
    planning_df
    .groupby(["term_id", "term", "week_start"], dropna=False)
    .agg(
        week_end=("date", "max"),
        special_dates=("label", lambda x: "; ".join(sorted(set(v for v in x if v)))),
        day_types=("day_type", lambda x: "; ".join(sorted(set(x))))
    )
    .reset_index()
)

week_summary = week_summary[
    week_summary["day_types"].str.contains("Instruction|Break|Study Day|Exam Period|Holiday", regex=True)
].copy()

def make_week_label(row):
    if "Break" in row["day_types"] and "Reading Week" in row["special_dates"]:
        return "Reading Week"
    if "Break" in row["day_types"]:
        return "Break"
    if "Exam Period" in row["day_types"]:
        return "Exam Period"
    if "Study Day" in row["day_types"]:
        return "Study Day"
    if "Holiday" in row["day_types"]:
        return "Holiday Week"
    if "Instruction" in row["day_types"]:
        return "Instruction Week"
    return "NA"

week_summary["week_label"] = week_summary.apply(make_week_label, axis=1)

week_summary = week_summary[
    ["term", "week_label", "week_start", "week_end", "day_types", "special_dates"]
]

week_summary["sort_date"] = pd.to_datetime(week_summary["week_start"])

week_summary = week_summary.sort_values(
    ["term", "sort_date"]
).reset_index(drop=True)

week_summary = week_summary[
    ["term", "week_label", "week_start", "week_end", "day_types", "special_dates"]
]

week_summary

In [ ]:
######### EXPORT 
from datetime import datetime

# Export calendar workbook with daily calendar, week summary, and metadata

version = config.get("version", 1)
status = config.get("status", "draft").upper()
academic_year = config["academic_year"].replace("-", "_")
generated_date = datetime.today().strftime("%Y-%m-%d")

# Choose draft or final output folder based on YAML status
OUTPUT_DIR = BASE_DIR / "outputs" / "calendars" / status.lower()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / f"academic_calendar_{academic_year}_v{version}_{status}_{generated_date}.xlsx"

metadata_df = pd.DataFrame([
    {"Field": "Academic Year", "Value": config.get("academic_year")},
    {"Field": "Version", "Value": version},
    {"Field": "Status", "Value": status},
    {"Field": "Last Reviewed", "Value": config.get("last_reviewed")},
    {"Field": "Generated Date", "Value": generated_date},
    {"Field": "Source File", "Value": str(DATA_FILE)},
])

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    metadata_df.to_excel(writer, sheet_name="Metadata", index=False)
    calendar_df.to_excel(writer, sheet_name="Daily Calendar", index=False)
    week_summary.to_excel(writer, sheet_name="Week Summary", index=False)

output_file